# Time-Series Forecasting with an LSTM (Jena Climate)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/river-kris/Keras-Timeseries/blob/main/notebooks/02_lstm_weather_forecasting.ipynb)

Based on the official Keras example
[Timeseries forecasting for weather prediction](https://keras.io/examples/timeseries/timeseries_weather_forecasting/)
(authors: Prabhanshu Attri, Yashika Sharma, Kristi Takach, Falak Shah). Data pipeline, model, and training code
match the official example. This notebook adds a `CONFIG` block (including options for stacked LSTMs and dropout),
seeding, timing, MAE reporting, and saving of results.

**How to use**
1. `Runtime > Change runtime type > T4 GPU`.
2. Edit `CONFIG` / `RUN_NAME` below (leave defaults for the baseline).
3. `Runtime > Run all`.
4. The last cell downloads `<RUN_NAME>.zip`. Unzip it into the repo's `results/` folder and commit.

## 1. Setup

In [ ]:
import os, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| Python:", platform.python_version())
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - switch to Runtime > Change runtime type > T4 GPU")

In [ ]:
def make_run_dir(run_name):
    run_dir = os.path.join("outputs", run_name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)


def env_info():
    gpus = tf.config.list_physical_devices("GPU")
    gpu_name = "none"
    if gpus:
        try:
            gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "gpu")
        except Exception:
            gpu_name = "gpu"
    return {"tensorflow": tf.__version__, "keras": keras.__version__, "gpu": gpu_name}


def plot_history(history_df, keys, title, path):
    plt.figure(figsize=(7, 4))
    for k in keys:
        if k in history_df:
            plt.plot(history_df["epoch"] + 1, history_df[k], label=k)
    plt.xlabel("epoch"); plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

## 2. Configuration
Defaults = official baseline (one LSTM layer with 32 units, 720 h of history sampled hourly,
predict temperature 72 steps (12 h) ahead).

Example experiments (change one thing at a time and set a new `RUN_NAME`):
| RUN_NAME | change |
|---|---|
| `lstm_units64` | `lstm_units: 64` |
| `lstm_stacked2` | `num_lstm_layers: 2` |
| `lstm_past360` | `past: 360` (shorter input window) |
| `lstm_lr3e-4` | `learning_rate: 3e-4` |
| `lstm_bs128` | `batch_size: 128` |
| `lstm_dropout0.2` | `dropout: 0.2` |

In [ ]:
RUN_NAME = "lstm_baseline"

CONFIG = {
    "seed": 42,
    # data / windowing (official values)
    "split_fraction": 0.715,
    "step": 6,          # sample one point per hour (data is every 10 min)
    "past": 720,        # use the last 720 timestamps (5 days) as input
    "future": 72,       # predict temperature 72 timestamps (12 h) ahead
    "feature_indices": [0, 1, 5, 7, 8, 10, 11],
    # model
    "lstm_units": 32,
    "num_lstm_layers": 1,   # >1 stacks LSTM layers (improvement option)
    "dropout": 0.0,         # dropout inside the LSTM layers (improvement option)
    # training
    "learning_rate": 0.001,
    "batch_size": 256,
    "epochs": 10,
    "early_stopping_patience": 5,
}

keras.utils.set_random_seed(CONFIG["seed"])
run_dir = make_run_dir(RUN_NAME)
print("Saving outputs to", run_dir)

## 3. Data: Jena Climate (2009-2016)
14 weather features recorded every 10 minutes at the Max Planck Institute for Biogeochemistry in Jena, Germany.

In [ ]:
from zipfile import ZipFile

uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = keras.utils.get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
zip_file = ZipFile(zip_path)
zip_file.extractall()
csv_path = "jena_climate_2009_2016.csv"

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
feature_keys = [
    "p (mbar)", "T (degC)", "Tpot (K)", "Tdew (degC)", "rh (%)", "VPmax (mbar)", "VPact (mbar)",
    "VPdef (mbar)", "sh (g/kg)", "H2OC (mmol/mol)", "rho (g/m**3)", "wv (m/s)", "max. wv (m/s)", "wd (deg)",
]
date_time_key = "Date Time"

# Correlation heatmap (as in the official example)
plt.figure(figsize=(9, 8))
plt.matshow(df[feature_keys].corr(), fignum=0)
plt.xticks(range(len(feature_keys)), feature_keys, rotation=90, fontsize=8)
plt.yticks(range(len(feature_keys)), feature_keys, fontsize=8)
plt.colorbar(); plt.title("Feature correlation", pad=80); plt.tight_layout()
plt.savefig(os.path.join(run_dir, "feature_correlation.png"), dpi=150); plt.show()

## 4. Preprocessing and windowing

In [ ]:
split_fraction = CONFIG["split_fraction"]
train_split = int(split_fraction * int(df.shape[0]))
step = CONFIG["step"]
past = CONFIG["past"]
future = CONFIG["future"]
learning_rate = CONFIG["learning_rate"]
batch_size = CONFIG["batch_size"]
epochs = CONFIG["epochs"]


def normalize(data, train_split):
    data_mean = data[:train_split].mean(axis=0)
    data_std = data[:train_split].std(axis=0)
    return (data - data_mean) / data_std, data_mean, data_std


selected_features = [feature_keys[i] for i in CONFIG["feature_indices"]]
print("Selected features:", selected_features)
n_features = len(selected_features)

features = df[selected_features]
features.index = df[date_time_key]
features, feat_mean, feat_std = normalize(features.values, train_split)
features = pd.DataFrame(features)

# Temperature is column 1 of the selected features; keep its std to convert errors back to degC
temp_std = float(feat_std[1])

train_data = features.loc[0 : train_split - 1]
val_data = features.loc[train_split:]

# Training dataset
start = past + future
end = start + train_split

x_train = train_data[[i for i in range(n_features)]].values
y_train = features.iloc[start:end][[1]]

sequence_length = int(past / step)

dataset_train = keras.utils.timeseries_dataset_from_array(
    x_train,
    y_train,
    sequence_length=sequence_length,
    sampling_rate=step,
    batch_size=batch_size,
)

# Validation dataset
x_end = len(val_data) - past - future
label_start = train_split + past + future

x_val = val_data.iloc[:x_end][[i for i in range(n_features)]].values
y_val = features.iloc[label_start:][[1]]

dataset_val = keras.utils.timeseries_dataset_from_array(
    x_val,
    y_val,
    sequence_length=sequence_length,
    sampling_rate=step,
    batch_size=batch_size,
)

for batch in dataset_train.take(1):
    inputs, targets = batch

print("Input shape:", inputs.numpy().shape)
print("Target shape:", targets.numpy().shape)

## 5. Model

In [ ]:
inputs = keras.layers.Input(shape=(inputs.shape[1], inputs.shape[2]))
x = inputs
for i in range(CONFIG["num_lstm_layers"]):
    last = i == CONFIG["num_lstm_layers"] - 1
    x = keras.layers.LSTM(
        CONFIG["lstm_units"], return_sequences=not last, dropout=CONFIG["dropout"]
    )(x)
outputs = keras.layers.Dense(1)(x)

model = keras.Model(inputs=inputs, outputs=outputs)
# "mae" is only an extra reported metric; the training loss is MSE as in the official example
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"]
)
model.summary()

## 6. Train

In [ ]:
path_checkpoint = os.path.join(run_dir, "model.weights.h5")
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=CONFIG["early_stopping_patience"]
)
modelckpt_callback = keras.callbacks.ModelCheckpoint(
    monitor="val_loss",
    filepath=path_checkpoint,
    verbose=1,
    save_weights_only=True,
    save_best_only=True,
)

t0 = time.time()
history = model.fit(
    dataset_train,
    epochs=epochs,
    validation_data=dataset_val,
    callbacks=[es_callback, modelckpt_callback, keras.callbacks.CSVLogger(os.path.join(run_dir, "history.csv"))],
)
train_time_s = time.time() - t0
print(f"Training time: {train_time_s/60:.1f} min")

## 7. Evaluate (best checkpoint on the validation set)

In [ ]:
model.load_weights(path_checkpoint)
val_mse, val_mae = model.evaluate(dataset_val, verbose=1)

hist = pd.DataFrame(history.history)
hist["epoch"] = np.arange(len(hist))
best_epoch = int(hist["val_loss"].idxmin())

metrics = {
    "run_name": RUN_NAME,
    "task": "forecasting",
    "model": "lstm",
    "best_val_mse": float(val_mse),
    "best_val_mae": float(val_mae),
    "best_val_mae_degC": float(val_mae * temp_std),   # MAE converted back to degrees Celsius
    "best_epoch": best_epoch + 1,
    "epochs_run": int(len(hist)),
    "final_train_mse": float(hist["loss"].iloc[-1]),
    "params": int(model.count_params()),
    "train_time_min": round(train_time_s / 60, 2),
    **env_info(),
}
print(json.dumps(metrics, indent=2))

In [ ]:
# Prediction examples (as in the official example), saved for the README
def show_plot(plot_data, delta, title, path):
    labels = ["History", "True Future", "Model Prediction"]
    marker = [".-", "rx", "go"]
    time_steps = list(range(-(plot_data[0].shape[0]), 0))
    future_x = delta if delta else 0
    plt.figure(figsize=(7, 4))
    plt.title(title)
    for i, val in enumerate(plot_data):
        if i:
            plt.plot(future_x, plot_data[i], marker[i], markersize=10, label=labels[i])
        else:
            plt.plot(time_steps, plot_data[i].flatten(), marker[i], label=labels[i])
    plt.legend(); plt.xlim([time_steps[0], (future_x + 5) * 2]); plt.xlabel("Time-Step")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()


for k, (x, y) in enumerate(dataset_val.take(3)):
    show_plot(
        [x[0][:, 1].numpy(), y[0].numpy(), model.predict(x, verbose=0)[0]],
        12,
        f"Single Step Prediction #{k+1}",
        os.path.join(run_dir, f"prediction_{k+1}.png"),
    )

## 8. Save results

In [ ]:
save_json(CONFIG, os.path.join(run_dir, "config.json"))
save_json(metrics, os.path.join(run_dir, "metrics.json"))
with open(os.path.join(run_dir, "model_summary.txt"), "w") as f:
    model.summary(print_fn=lambda s, **kw: f.write(s + "\n"))

plot_history(hist, ["loss", "val_loss"], f"{RUN_NAME}: MSE loss", os.path.join(run_dir, "loss.png"))
print(os.listdir(run_dir))

In [ ]:
# Zip this run's outputs and download them.
# Unzip into the repo so the files end up at results/<RUN_NAME>/...
import shutil
archive = shutil.make_archive(RUN_NAME, "zip", root_dir="outputs", base_dir=RUN_NAME)
print("Created", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab - copy", run_dir, "into results/ manually.")

## 9. Observations
_Write what you noticed: train vs. validation loss, whether it was still improving at epoch 10,
validation MAE in degC, training time, how the predictions look._